# 실습 9: Ollama LLM을 활용한 HTTP 분류 (2교시)

특성 추출 없이 **HTTP 요청 텍스트를 그대로** Ollama gemma3:4b에 보여주고
정상/공격을 분류합니다.

**사전 조건**:
- Ollama 서버 실행 중 (`ollama serve`)
- `gemma3:4b` 모델 다운로드됨 (`ollama list`로 확인)
- 7주차 `processed_data.pkl` 존재 (LLM용 텍스트 샘플 포함)


In [12]:
# %% [Setup] 패키지 import 및 LLM용 샘플 로드
import pickle
import time
import json
import re
import pandas as pd
from urllib.parse import unquote
from sklearn.metrics import accuracy_score, f1_score, classification_report

import ollama  # pip install ollama

with open("processed_data.pkl", "rb") as f:
    data = pickle.load(f)

llm_sample = data["llm_sample"].head(100).reset_index(drop=True)
print(f"분류 대상: {len(llm_sample)}건")
print(f"라벨 분포: 정상 {(llm_sample.get('is_attack',0)==0).sum()}건 / "
      f"공격 {(llm_sample.get('is_attack',0)==1).sum()}건")


분류 대상: 100건
라벨 분포: 정상 56건 / 공격 44건


## 1. 분류 프롬프트 설계

LLM 응답을 안정적으로 파싱하기 위해:
- **Few-shot 예시** 2개로 출력 형식을 학습시킴
- **JSON 형태**로 응답하도록 강제
- 영어 프롬프트(gemma3:4b가 영어에 더 정확)

In [13]:
# %% [1] HTTP 텍스트 재구성 + 프롬프트 함수
import re, json

def build_http_text(row) -> str:
    method = row.get("method", "GET")
    url    = unquote(str(row.get("url", "")), encoding="latin-1")
    body   = str(row.get("body_decoded", row.get("body", "")) or "")
    text   = f"{method} {url} HTTP/1.1"
    if body and body != "nan":
        text += f"\nBody: {body[:200]}"
    return text


PROMPT_TEMPLATE = """\
You are a WAF (Web Application Firewall) rule engine analyzing HTTP requests.
Classify the request as EXACTLY "Normal" or "Anomalous".

### Attack patterns to detect:
- SQL Injection: UNION, SELECT, OR '1'='1, --, ;DROP, SLEEP(), BENCHMARK()
- XSS: <script>, javascript:, onerror=, onload=, alert(, document.cookie
- LFI/Path Traversal: ../, ../../, /etc/passwd, /proc/self
- Command Injection: ;ls, |whoami, `id`, &&cat, $(cmd)
- SSRF: file://, dict://, gopher://, internal IP ranges (127.x, 192.168.x, 10.x)
- XXE: <!ENTITY, SYSTEM "file://
- Log4Shell: ${{jndi:, ${{::-j, ${{lower:j}}
- Scanner signatures: nikto, sqlmap, nessus, masscan in User-Agent

### Normal patterns:
- Standard GET for static resources (.html, .js, .css, .png, .jpg)
- Form POSTs with normal alphanumeric parameters
- REST API calls with valid path structures
- Encoded characters that decode to benign strings

### Output rules (STRICT):
- Respond with ONLY a JSON object, no extra text
- JSON must have exactly two keys: "label" and "reason"
- "label" must be exactly "Normal" or "Anomalous" (case-sensitive)
- "reason" must be one concise sentence (max 20 words) citing the specific pattern found

### Examples:
Request: GET /images/logo.png HTTP/1.1
{{"label": "Normal", "reason": "Static image resource request with no suspicious parameters."}}

Request: GET /search?q=' UNION SELECT username,password FROM users-- HTTP/1.1
{{"label": "Anomalous", "reason": "SQL Injection: UNION SELECT targeting credentials table with comment terminator."}}

Request: GET /page?file=../../etc/passwd HTTP/1.1
{{"label": "Anomalous", "reason": "Path Traversal: directory escape sequences targeting /etc/passwd."}}

Request: POST /login HTTP/1.1
Body: username=admin&password=P%40ssw0rd123
{{"label": "Normal", "reason": "Standard login form submission with encoded but benign credentials."}}

Request: GET /index.php?cmd=;cat+/etc/shadow HTTP/1.1
{{"label": "Anomalous", "reason": "Command Injection: semicolon-separated cat command targeting shadow password file."}}

Now classify this request:
Request: {http_text}
"""


def classify_with_llm(http_text: str, model: str = "gemma3:4b") -> dict:
    """Ollama로 HTTP 요청 분류 -> {label, reason}"""
    prompt = PROMPT_TEMPLATE.format(http_text=http_text[:300])  # 길이 제한

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0,
            "num_predict": 80,  # JSON만 생성하도록 출력 제한
        },
    )
    text = response["message"]["content"].strip()

    # 1차: 전체가 JSON인 경우
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2차: JSON 블록 추출
    match = re.search(r"\{[^{}]*\"label\"[^{}]*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    # 3차: 키워드 휴리스틱 (Unknown 최소화)
    label = "Unknown"
    if re.search(r'\b(anomalous|attack|malicious|suspicious)\b', text, re.I):
        label = "Anomalous"
    elif re.search(r'\b(normal|benign|legitimate|safe)\b', text, re.I):
        label = "Normal"

    return {"label": label, "reason": text[:100]}


# 단건 테스트
test_text = "GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1"
print("입력:", test_text)
print("응답:", classify_with_llm(test_text))

입력: GET /tienda1/publico/anadir.jsp?id=2'+OR+'1'='1 HTTP/1.1
응답: {'label': 'Anomalous', 'reason': 'SQL Injection: The request contains an OR condition to bypass authentication.'}


## 2. 100건 분류 + 시간 측정

CPU 환경 기준 건당 1~3초가 표준. 100건 ≈ 2~5분 소요.

In [14]:
# %% [2] 100건 분류
results = []
start = time.time()

for i, row in llm_sample.iterrows():
    http_text = build_http_text(row)
    result = classify_with_llm(http_text)
    true_label = "Anomalous" if row.get("is_attack", 0) == 1 else "Normal"
    results.append({
        "idx": i,
        "true": true_label,
        "pred": result.get("label", "Unknown"),
        "reason": result.get("reason", "")[:120],
        "http_short": http_text[:100],
    })
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/{len(llm_sample)}건 완료 "
              f"({elapsed:.1f}초, 건당 {elapsed/(i+1):.2f}초)")

llm_time = time.time() - start
llm_df = pd.DataFrame(results)
print(f"\n총 소요: {llm_time:.1f}초")
print(f"1만 건 환산: 약 {llm_time/100*10000/60:.0f}분")


  10/100건 완료 (4.7초, 건당 0.47초)
  20/100건 완료 (8.7초, 건당 0.43초)
  30/100건 완료 (13.3초, 건당 0.44초)
  40/100건 완료 (17.9초, 건당 0.45초)
  50/100건 완료 (22.2초, 건당 0.44초)
  60/100건 완료 (26.5초, 건당 0.44초)
  70/100건 완료 (31.1초, 건당 0.44초)
  80/100건 완료 (35.9초, 건당 0.45초)
  90/100건 완료 (40.4초, 건당 0.45초)
  100/100건 완료 (44.7초, 건당 0.45초)

총 소요: 44.7초
1만 건 환산: 약 75분


In [15]:
# %% [3] 정확도/F1 계산
llm_df["pred_clean"] = llm_df["pred"].replace({"Unknown":"Normal"})
y_true = (llm_df["true"] == "Anomalous").astype(int)
y_pred = (llm_df["pred_clean"] == "Anomalous").astype(int)

llm_acc = accuracy_score(y_true, y_pred)
llm_f1  = f1_score(y_true, y_pred)

print(f"LLM 정확도: {llm_acc:.4f}")
print(f"LLM F1:    {llm_f1:.4f}")
print(f"분류 실패(Unknown): {(llm_df['pred']=='Unknown').sum()}건")
print()
print(classification_report(y_true, y_pred, target_names=["Normal","Anomalous"]))


LLM 정확도: 0.7000
LLM F1:    0.6739
분류 실패(Unknown): 0건

              precision    recall  f1-score   support

      Normal       0.75      0.70      0.72        56
   Anomalous       0.65      0.70      0.67        44

    accuracy                           0.70       100
   macro avg       0.70      0.70      0.70       100
weighted avg       0.70      0.70      0.70       100



## 3. 자연어 판단 근거 검토 ★

LLM의 가장 큰 강점: **왜 그렇게 판단했는지** 사람이 읽을 수 있는 문장으로 설명.
이는 SOC(보안관제) 분석가가 1차 분류를 검토할 때 매우 유용합니다.

In [16]:
# %% [4] 공격으로 판단한 사례 + LLM 근거
print("=== LLM이 공격으로 판단한 사례 (상위 5건) ===\n")
attack_pred = llm_df[llm_df["pred"] == "Anomalous"].head(5)
for _, r in attack_pred.iterrows():
    correct = "OK" if r["true"] == "Anomalous" else "오탐"
    print(f"[{correct}] 실제={r['true']:10s}  요청: {r['http_short']}")
    print(f"   - LLM 근거: {r['reason']}\n")


=== LLM이 공격으로 판단한 사례 (상위 5건) ===

[OK] 실제=Anomalous   요청: GET /tienda1/miembros/editar.jsp?modo=registro&loginA=lieure&password=rEbatible&nombre=Tarciano&apel
   - LLM 근거: Encoded characters and unusual URL parameters suggest potential manipulation.

[오탐] 실제=Normal      요청: POST /tienda1/publico/entrar.jsp HTTP/1.1
Body: errorMsg=Credenciales+incorrectas
   - LLM 근거: Form POST with 'errorMsg' parameter containing potentially malicious string.

[OK] 실제=Anomalous   요청: GET /tienda1/asf-logo-wide.gif/ HTTP/1.1
   - LLM 근거: Path Traversal: malformed URL path attempting to access a non-existent resource.

[OK] 실제=Anomalous   요청: GET /tienda1/miembros/imagenes/.inc HTTP/1.1
   - LLM 근거: Path Traversal: Attempting to access a .inc file outside the intended directory.

[OK] 실제=Anomalous   요청: POST /tienda1/publico/vaciar.jsp HTTP/1.1
Body: B2=Vaciar+carrito%3CSCRIPT%3Ealert%28%22Paros%22%29%
   - LLM 근거: XSS: JavaScript alert payload detected within the JSp POST body.



In [7]:
# %% [5] 결과 저장 (3교시에서도 활용)
with open("llm_classification_results.pkl", "wb") as f:
    pickle.dump({
        "llm_df": llm_df,
        "llm_acc": llm_acc,
        "llm_f1": llm_f1,
        "llm_time": llm_time,
        "n_samples": len(llm_sample),
    }, f)
print(">> llm_classification_results.pkl 저장 완료")


>> llm_classification_results.pkl 저장 완료


**다음**: `comparison_analysis.ipynb`로 1교시 ML 결과와 종합 비교합니다.